# Sensor Placement Optimisation (MuJoCo)

Optimises the **sensor type** assigned to **5 fixed vehicle mount slots** using CMA-ES.

```
Slots: top_front_edge_l  |  top_front_edge_r  |  front_face_center
       side_front_edge_l  |  side_front_edge_r
```

**Run order (read before starting)**
1. Run **Cell 1** (CMA ABI fix) → `Runtime → Restart session`.
2. Run **Cells 2 – 4** (clone, install, verify imports).
3. Edit **Cell 4** (user config) to taste — `CMA_POPSIZE` / `CMA_POPSIZE_BY_MODE` control population size.
4. Run patch cells (**obstacles → env manager rewrite**) — these rewrite cloned source files; run once per session.
5. Run the **experiment cell** (labeled Cell 9 in code): runs **two consecutive** CMA-ES jobs — `safety_first` then `efficiency` (~2× runtime vs a single mode).
6. Run the **next cell** (Cell 10): overlays both runs on **one** convergence plot. `results/_notebook_dual_run_manifest.json` records which folder is which mode.
7. Run later cells for optional single-run introspection / video (**video uses the newest result folder**, i.e. `efficiency` after a full dual run unless you change it).

> **Tip — quick smoke test:** set `MAX_GENERATIONS = 5` and `N_EPISODES = 3` in **Cell 4** before your first run.

In [1]:
# ── Cell 1 ─ CMA / NumPy ABI fix ────────────────────────────────────────────
# Run this cell ONCE, then do Runtime → Restart session before continuing.
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall", "--no-cache-dir", "cma",
])
print("✅ cma reinstalled — do Runtime → Restart session NOW, then run Cell 2 onwards.")

✅ cma reinstalled — do Runtime → Restart session NOW, then run Cell 2 onwards.


In [5]:
# ── Cell 2 ─ Clone repo + install requirements ───────────────────────────────
import os, shutil, subprocess, sys
from pathlib import Path

WORK   = Path("/content")
TARGET = Path("/content/sensor_placement_opt_MUJOCO_ver")

if not (TARGET / "sensor_opt").is_dir():
    if TARGET.exists():
        shutil.rmtree(TARGET, ignore_errors=True)
    os.chdir(WORK)
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/dcaglar-28/sensor_placement_opt_MUJOCO_ver.git",
        str(TARGET.name),
    ])

os.chdir(TARGET)
if str(TARGET) not in sys.path:
    sys.path.insert(0, str(TARGET))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
print("cwd:", os.getcwd())

cwd: /content/sensor_placement_opt_MUJOCO_ver


In [6]:
# ── Cell 3 ─ Verify import stack ─────────────────────────────────────────────
import os, sys
from pathlib import Path

TARGET = Path("/content/sensor_placement_opt_MUJOCO_ver")
os.chdir(TARGET)
if str(TARGET) not in sys.path:
    sys.path.insert(0, str(TARGET))

import cma, jax, matplotlib, mlflow, mujoco
import numpy as np, pandas as pd, rich, scipy, sklearn, torch, yaml
import sensor_opt

print("✅ All imports OK — MuJoCo", mujoco.__version__)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ── Cell 4 ─ USER CONFIGURATION (edit this cell only) ────────────────────────
#
# OPTIMIZATION_MODE — used for Cell 4 summary print and LOSS_WEIGHT_OVERRIDES below.
# The experiment cell runs BOTH presets sequentially (safety_first, then efficiency);
# it does not use this variable for those runs (see OPTIMIZATION_MODES_TO_RUN there).
#   "safety_first" — minimise blind-spots + collisions; bandwidth cost is a tiebreaker
#   "efficiency"   — still safety-heavy, but strongly penalises high-bandwidth sensors
#
OPTIMIZATION_MODE = "safety_first"

# ── Scenario ──────────────────────────────────────────────────────────────────
TRIAL_TYPE        = "multi_objective"
PATH_LENGTH_M     = 15.0
VEHICLE_SPEED_MPS = 5.0
BASE_RANDOM_SEED  = 42
MAX_GENERATIONS   = 100    # 5–25 smoke-test; 50–100 for real results
N_EPISODES        = 3

# CMA-ES population (requires use_recommended_popsize=False in experiment cell).
CMA_POPSIZE       = 20
# Optional per-mode population for the dual run; missing keys fall back to CMA_POPSIZE.
CMA_POPSIZE_BY_MODE = {
    "safety_first": CMA_POPSIZE,
    "efficiency": CMA_POPSIZE,
}

# ── Hardware platform ─────────────────────────────────────────────────────────
# Enter your chip + memory specs here. Bandwidth is the key constraint on
# unified-memory chips (M-series, Snapdragon X, etc.) because CPU, GPU, and
# Neural Engine all share the same pool.
HARDWARE_PLATFORM = {
    "name":                  "Apple M5 (24 GB)",
    "total_memory_gb":       24,
    "memory_bandwidth_gbps": 153.6,   # LPDDR5X @ 9600 MT/s
    "neural_engine_tops":    38,      # used for latency scaling (future)
}


# "jetson_orin": {
#         "name":                  "NVIDIA Jetson Orin (32 GB)",
#         "total_memory_gb":       32,
#         "memory_bandwidth_gbps": 204.8,   # LPDDR5 @ 6400 MT/s ×2 channels
#         "neural_engine_tops":    275,     # NVDLA + GPU tensor cores
#     }

# ── Sensor specs — enter values from your datasheets ─────────────────────────
# data_rate_mbps : raw data the sensor streams per active instance
# fov_deg        : horizontal field-of-view
# max_range_m    : detection range
# latency_s      : processing latency (sensor → usable output)
# n_rays         : ray-cast density used in MuJoCo simulation
# max_count      : how many physical units you actually have available
#
# These values flow automatically into:
#   • the MuJoCo yaml patch (Cell 7) — so DO NOT edit sensor params there
#   • the cost function         (bandwidth fraction replaces USD cost)
#   • the per-type availability cap

SENSOR_SPECS = {
    "lidar": {
        "data_rate_mbps":  250,   # 128-ch lidar ~250 Mbps
        "fov_deg":         120,
        "max_range_m":      20,
        "latency_s":       0.10,
        "n_rays":           64,
        "max_count":         2,   # units physically available
    },
    "radar": {
        "data_rate_mbps":    5,   # radar output is small
        "fov_deg":          90,
        "max_range_m":      20,
        "latency_s":        0.05,
        "n_rays":           32,
        "max_count":         3,
    },
    "camera": {
        "data_rate_mbps":   40,   # 1080p30 compressed
        "fov_deg":          60,
        "max_range_m":      10,
        "latency_s":        0.033,
        "n_rays":           16,
        "max_count":         5,
    },
    "disabled": {
        "data_rate_mbps":    0,
        "fov_deg":           0,
        "max_range_m":       0,
        "latency_s":         0.0,
        "n_rays":            0,
        "max_count":        99,
    },
}

# ── Derived values (computed automatically — do not edit below this line) ─────

# Cost = bandwidth fraction × 10 000 (integer-safe, dimensionless)
# This replaces the old USD cost. The optimizer penalises sensors that eat
# a larger share of the chip's memory bandwidth.
_bw_mbps = HARDWARE_PLATFORM["memory_bandwidth_gbps"] * 1_000
SENSOR_COSTS_USD = {          # name kept for back-compat with the run cell
    k: int((v["data_rate_mbps"] / _bw_mbps) * 10_000)
    for k, v in SENSOR_SPECS.items()
}

# Per-type availability caps fed directly from SENSOR_SPECS
MAX_SENSOR_COUNTS = {k: v["max_count"] for k, v in SENSOR_SPECS.items()}
MAX_SENSOR_COUNT  = None   # legacy single-cap; superseded by MAX_SENSOR_COUNTS

# Total budget cap expressed in the same bandwidth-fraction units
# (10 000 = 100% of bandwidth; 5 simultaneous sensors shouldn't exceed ~50%)
# Budget cap expressed in bandwidth-fraction cost units (same scale as SENSOR_COSTS_USD).
# MAX_BANDWIDTH_FRACTION = fraction of total chip bandwidth sensors may consume.
# 0.30 → sensors collectively may use at most 30% of the M5's 153.6 GB/s.
# Raise this if you want the optimizer to consider more/heavier sensors.
MAX_BANDWIDTH_FRACTION  = 0.30
MAX_HARDWARE_BUDGET_USD = int(MAX_BANDWIDTH_FRACTION * 10_000)  # e.g. 0.30 → 3000 cost units

# ── Loss weight presets ───────────────────────────────────────────────────────
_LOSS_PRESETS = {
    # Preset 1 — Safety first
    # Blind-spot coverage and detection latency dominate; bandwidth cost is
    # only a tiebreaker between otherwise equivalent configs.
    "safety_first": {
        "w_acc":  0.60,   # detection rate  ← primary
        "w_lat":  0.35,   # latency         ← secondary
        "w_cost": 0.05,   # bandwidth cost  ← tiebreaker
    },
    # Preset 2 — Efficiency
    # Still safety-heavy, but now 30% of the loss comes from bandwidth cost.
    # Will push the optimizer toward cameras + radar over lidar.
    "efficiency": {
        "w_acc":  0.45,
        "w_lat":  0.25,
        "w_cost": 0.30,   # strong bandwidth pressure
    },
}

assert OPTIMIZATION_MODE in _LOSS_PRESETS, (
    f"Unknown OPTIMIZATION_MODE '{OPTIMIZATION_MODE}'. "
    f"Choose from: {list(_LOSS_PRESETS)}"
)
LOSS_WEIGHT_OVERRIDES = _LOSS_PRESETS[OPTIMIZATION_MODE]

# ── Derived scenario scaling (computed automatically — do not edit) ───────────
# Obstacle density stays at 1 per metre regardless of path length
N_OBSTACLES = int(PATH_LENGTH_M * 1.0)

# Lateral scatter widens with speed so side sensors stay genuinely relevant.
# Baseline: ±1.5 m at 5 m/s. Capped at ±3.0 m.
_LATERAL_M = round(min(1.5 * (VEHICLE_SPEED_MPS / 5.0), 3.0), 2)

# ── Summary ───────────────────────────────────────────────────────────────────
_bw_pct = {k: f"{v/100:.2f}%" for k, v in SENSOR_COSTS_USD.items()}
print(f"Platform       : {HARDWARE_PLATFORM['name']}")
print(f"Bandwidth      : {HARDWARE_PLATFORM['memory_bandwidth_gbps']} GB/s  |  Memory: {HARDWARE_PLATFORM['total_memory_gb']} GB")
print(f"Optimization   : {OPTIMIZATION_MODE.upper()}")
print(f"Loss weights   : {LOSS_WEIGHT_OVERRIDES}")
print(f"Bandwidth cost : {_bw_pct}  (fraction of {HARDWARE_PLATFORM['memory_bandwidth_gbps']} GB/s)")
print(f"Sensor caps    : { {k: v['max_count'] for k,v in SENSOR_SPECS.items() if k != 'disabled'} }")
print(f"Generations    : {MAX_GENERATIONS}  |  Episodes/eval: {N_EPISODES}")
print(f"CMA popsize    : {CMA_POPSIZE}  |  BY_MODE: {CMA_POPSIZE_BY_MODE}")
print("Ready — run patch cells, then Cell 9 (dual optimisation).")


In [4]:
# ── Cell 5 ─ Patch obstacles.py ──────────────────────────────────────────────
# lateral_m and N_OBSTACLES are derived from Cell 4 (PATH_LENGTH_M, VEHICLE_SPEED_MPS).
# Must always run AFTER Cell 4.
from pathlib import Path

dest = Path('/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/simulation/obstacles.py')

_TEMPLATE = '"""Obstacle layouts for the MuJoCo sensor-placement simulation."""\nfrom __future__ import annotations\nfrom typing import List, Tuple\nimport numpy as np\n\nPosition = Tuple[float, float, float]\n\nFIXED_SEEDS: List[int] = [42_000, 43_000, 44_000]\n\n\ndef generate_obstacles(\n    n_obstacles: int,\n    path_length_m: float,\n    rng_seed: int,\n    lateral_m: float = LAT_PLACEHOLDER,\n) -> List[Position]:\n    """Return (x, y, z) obstacle positions.\n\n    x -- uniformly sampled in [2.0, path_length_m - 1.0]\n    y -- uniformly sampled in [-LAT_PLACEHOLDER, LAT_PLACEHOLDER] m\n    z -- fixed at 0.15 m\n    """\n    n = int(max(0, n_obstacles))\n    lo_x = 2.0\n    hi_x = max(lo_x, float(path_length_m) - 1.0)\n    rng = np.random.default_rng(int(rng_seed))\n    return [\n        (float(rng.uniform(lo_x, hi_x)), float(rng.uniform(-lateral_m, lateral_m)), 0.15)\n        for _ in range(n)\n    ]\n\n\ndef get_fixed_seeds() -> List[int]:\n    """Convenience accessor for the canonical seed list."""\n    return list(FIXED_SEEDS)\n'

src = _TEMPLATE.replace('LAT_PLACEHOLDER', str(_LATERAL_M))
dest.write_text(src, encoding='utf-8')

print(f'✅ obstacles.py patched — ±{_LATERAL_M} m lateral | {N_OBSTACLES} obstacles | 3 fixed seeds')
for line in dest.read_text().split('\n'):
    if 'lateral_m' in line:
        print(f'   {line}')


✅ obstacles.py patched — ±1.5 m lateral | 15 obstacles | 3 fixed seeds
       lateral_m: float = 1.5,
           (float(rng.uniform(lo_x, hi_x)), float(rng.uniform(-lateral_m, lateral_m)), 0.15)


In [5]:
# ── Cell 6 ─ Patch mjcf.py ───────────────────────────────────────────────────
# Shrink obstacle sphere radius 0.25 → 0.15 m so that:
#   • spheres sit on the ground at z=0.15 (centre height == radius)
#   • smaller targets make sensor type / FOV differences more meaningful
from pathlib import Path

mjcf_path = Path("/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/simulation/mjcf.py")
src = mjcf_path.read_text(encoding="utf-8")

if 'size="0.15"' in src:
    print("ℹ️  mjcf.py already patched — skipping.")
elif 'size="0.25"' in src:
    src = src.replace('size="0.25"', 'size="0.15"')
    mjcf_path.write_text(src, encoding="utf-8")
    print("✅ mjcf.py patched — obstacle radius 0.25 → 0.15 m")
else:
    print("⚠️  Expected size=\"0.25\" not found in mjcf.py — inspect manually.")
    !grep -n 'size=' {mjcf_path} | head -20

ℹ️  mjcf.py already patched — skipping.


In [6]:
# ── Cell 7 ─ Patch mujoco_vehicle.yaml ───────────────────────────────────────
# Sensor params (fov, range, latency, n_rays) are now pulled from SENSOR_SPECS
# defined in Cell 4. Do NOT edit sensor values here directly.
import yaml
from pathlib import Path

cfg_path = Path("/content/sensor_placement_opt_MUJOCO_ver/configs/mujoco_vehicle.yaml")
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

# Build sensor_overrides from SENSOR_SPECS (set in Cell 4)
sensor_overrides = {
    k: {
        "n_rays":      v["n_rays"],
        "fov_deg":     v["fov_deg"],
        "max_range_m": v["max_range_m"],
        "latency_s":   v["latency_s"],
    }
    for k, v in SENSOR_SPECS.items()
    if k != "disabled"
}

for sensor, params in sensor_overrides.items():
    cfg.setdefault("sensor_models", {}).setdefault(sensor, {}).update(params)

cfg_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False), encoding="utf-8")
print(f"✅ mujoco_vehicle.yaml patched from SENSOR_SPECS ({HARDWARE_PLATFORM['name']})")
print("   Sensor models applied:")
for s, p in sensor_overrides.items():
    print(f"     {s:8s}  fov={p['fov_deg']:>4}°  range={p['max_range_m']:>4}m  "
          f"latency={p['latency_s']:.3f}s  rays={p['n_rays']}")
!grep -A6 'sensor_models:' {cfg_path}


✅ mujoco_vehicle.yaml patched from SENSOR_SPECS (Apple M5 (24 GB))
   Sensor models applied:
     lidar     fov= 120°  range=  20m  latency=0.100s  rays=64
     radar     fov=  90°  range=  20m  latency=0.050s  rays=32
     camera    fov=  60°  range=  10m  latency=0.033s  rays=16
sensor_models:
  lidar:
    cost_usd: 2000
    range_m: 25
    horizontal_fov_deg: 120
    vertical_fov_deg: 120
    latency_ms: 100


In [7]:
# ── Cell 8 ─ Patch mujoco_env_manager.py ─────────────────────────────────────
# Cycles episodes through the same 3 fixed seeds defined in obstacles.py,
# replacing the original per-generation variable seed.
#
# Fixes vs original:
#   • _FIXED_SEEDS list matches obstacles.py exactly (3 seeds, not 6)
#   • safe replace: checks for already-patched state before asserting
from pathlib import Path

env_path = Path(
    "/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/inner_loop/mujoco_env_manager.py"
)
src = env_path.read_text(encoding="utf-8")

_OLD = """            pos = generate_obstacles(
                self.n_obstacles,
                self.path_length_m,
                gen_seed + ep,
            )"""

# Must match FIXED_SEEDS in obstacles.py (3 entries).
_NEW = """            # Cycle through 3 fixed obstacle layouts for deterministic evaluation.
            # Seed list must stay in sync with FIXED_SEEDS in simulation/obstacles.py.
            _FIXED_SEEDS = [42_000, 43_000, 44_000]
            ep_seed = _FIXED_SEEDS[ep % len(_FIXED_SEEDS)]
            pos = generate_obstacles(
                self.n_obstacles,
                self.path_length_m,
                ep_seed,
            )"""

_SENTINEL = "_FIXED_SEEDS = [42_000, 43_000, 44_000]"

if _SENTINEL in src:
    print("ℹ️  mujoco_env_manager.py already patched — skipping.")
elif _OLD in src:
    src = src.replace(_OLD, _NEW)
    env_path.write_text(src, encoding="utf-8")
    print("✅ mujoco_env_manager.py patched — 3-seed episode cycling.")
else:
    print("⚠️  Target pattern not found — the upstream file may have changed.")
    print("    Inspect the file and update _OLD to match:")
    !grep -n 'gen_seed\|generate_obstacles' {env_path} | head -20

print("Verifying:")
!grep -A5 '_FIXED_SEEDS' {env_path} | head -15

⚠️  Target pattern not found — the upstream file may have changed.
    Inspect the file and update _OLD to match:
23:from sensor_opt.simulation.obstacles import generate_obstacles
171:            pos = generate_obstacles(
Verifying:


In [8]:
# ── Fix: remove get_generation_seed from mujoco_env_manager.py ───────────────
from pathlib import Path

path = Path("/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/inner_loop/mujoco_env_manager.py")
src = path.read_text(encoding="utf-8")
print("BEFORE (import line):")
for line in src.splitlines():
    if "get_generation_seed" in line:
        print(" ", line)

# Fix the import line
fixed = src.replace(
    "from sensor_opt.simulation.obstacles import generate_obstacles, get_generation_seed",
    "from sensor_opt.simulation.obstacles import generate_obstacles",
).replace(
    "from sensor_opt.simulation.obstacles import get_generation_seed, generate_obstacles",
    "from sensor_opt.simulation.obstacles import generate_obstacles",
)

# Fix any call sites — replace get_generation_seed(...) with the rng seed directly
# The patched obstacles.py cycles seeds via _FIXED_SEEDS in env_manager itself,
# so any call like get_generation_seed(gen) should become the inline equivalent.
# Most common pattern: seed = get_generation_seed(generation) → remove or inline.
import re
fixed = re.sub(r'\bget_generation_seed\b', '_get_generation_seed_REMOVED', fixed)

path.write_text(fixed, encoding="utf-8")
print("\nAFTER — remaining references:")
remaining = [l for l in fixed.splitlines() if "get_generation_seed" in l]
print("\n".join(remaining) if remaining else "  None ✅")
print("\n✅ Done — but check remaining references above before running Cell 9.")

BEFORE (import line):

AFTER — remaining references:
  None ✅

✅ Done — but check remaining references above before running Cell 9.


In [9]:
# ── Audit: find ALL remaining get_generation_seed references ─────────────────
import subprocess
result = subprocess.run(
    ["grep", "-rn", "get_generation_seed",
     "/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/"],
    capture_output=True, text=True
)
print(result.stdout or "✅ No references remain — safe to run Cell 9.")

✅ No references remain — safe to run Cell 9.


In [10]:
# ── Inspect the call site ─────────────────────────────────────────────────────
from pathlib import Path

src = Path("/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/inner_loop/mujoco_env_manager.py").read_text()
lines = src.splitlines()
for i, line in enumerate(lines):
    if "_get_generation_seed_REMOVED" in line or "_FIXED_SEEDS" in line or "gen_seed" in line:
        start = max(0, i - 3)
        end = min(len(lines), i + 5)
        print(f"── line {i+1} ──")
        print("\n".join(f"{j+1}: {lines[j]}" for j in range(start, end)))
        print()

In [11]:
# ── Fix both files ────────────────────────────────────────────────────────────
import re
from pathlib import Path

# 1. __init__.py — remove from __all__ / export list
init_path = Path("/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/simulation/__init__.py")
src = init_path.read_text(encoding="utf-8")
fixed = re.sub(r',?\s*["\']get_generation_seed["\']', "", src)
fixed = re.sub(r'["\']get_generation_seed["\'],?\s*', "", fixed)
init_path.write_text(fixed, encoding="utf-8")
print("✅ __init__.py cleaned")

# 2. mujoco_env_manager.py — replace the removed call with inline seed logic.
#    The original get_generation_seed(gen, base_seed) almost certainly just did:
#      base_seed + gen   (or a simple deterministic combination)
#    Cell 8's patch uses _FIXED_SEEDS cycling, so we replace the call with that.
mgr_path = Path("/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/inner_loop/mujoco_env_manager.py")
src = mgr_path.read_text(encoding="utf-8")

# Replace the broken call with inline seed derived from _FIXED_SEEDS cycling
fixed = src.replace(
    "gen_seed = _get_generation_seed_REMOVED(self._gen, self.base_random_seed)",
    "gen_seed = _FIXED_SEEDS[self._gen % len(_FIXED_SEEDS)]",
)

mgr_path.write_text(fixed, encoding="utf-8")
print("✅ mujoco_env_manager.py call site fixed")

# 3. Final audit
import subprocess
result = subprocess.run(
    ["grep", "-rn", "get_generation_seed",
     "/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/"],
    capture_output=True, text=True
)
print("\nRemaining references:")
print(result.stdout or "  None ✅ — safe to run Cell 9.")

✅ __init__.py cleaned
✅ mujoco_env_manager.py call site fixed

Remaining references:
  None ✅ — safe to run Cell 9.


In [12]:
# ── Complete rewrite of mujoco_env_manager.py with CRN ───────────────────────
from pathlib import Path

NEW_SRC = '''"""
MuJoCo: kinematic vehicle on +X, five fixed sites, mocap obstacle pool.
Contacts vehicle–obstacles are excluded in MJCF. Geometry via mj_kinematics (no mj_step in episode).

Seed strategy — Common Random Numbers (CRN):
  - Each generation draws a fresh pool of N_CRN_SEEDS seeds derived from
    (base_random_seed, generation). Every candidate in that generation is
    evaluated on the SAME pool, making within-generation loss comparisons
    fair (apples-to-apples). Across generations the pool rotates, so the
    optimizer cannot overfit to a fixed set of obstacle layouts.
"""

from __future__ import annotations

from typing import Any, Dict, List

import numpy as np

from sensor_opt.encoding.config import SensorConfig
from sensor_opt.loss.loss import EvalMetrics
from sensor_opt.simulation.mjcf import SLOT_NAMES, build_vehicle_mjcf
from sensor_opt.simulation.mujoco_runner import run_episode
from sensor_opt.simulation.obstacles import generate_obstacles
from sensor_opt.simulation.sensor_specs import get_sensor_specs

try:
    import mujoco
except ImportError:  # pragma: no cover
    mujoco = None  # type: ignore[assignment]

# Number of distinct obstacle layouts sampled per generation.
# 10 is enough to reduce layout variance while keeping evaluation fast.
# Increase to 20-50 for final/publication runs.
N_CRN_SEEDS = 10


def _crn_seeds(base_seed: int, generation: int, n: int) -> list[int]:
    """Return n deterministic seeds for this generation using CRN.

    Seeds are derived from (base_seed, generation) so they are:
      - identical for all candidates within a generation (fair comparison)
      - different across generations (no cross-generation overfitting)
    """
    rng = np.random.default_rng([int(base_seed), int(generation)])
    return [int(s) for s in rng.integers(0, 2**31 - 1, size=n)]


def _slot_to_config(config: SensorConfig) -> Dict[str, str]:
    out: Dict[str, str] = {s: "disabled" for s in SLOT_NAMES}
    for s in config.sensors:
        if s.slot in out:
            out[str(s.slot)] = str(s.sensor_type) if s.is_active() else "disabled"
    return out


class MujocoEnvManager:
    def __init__(
        self,
        *,
        num_envs: int = 1,
        n_obstacles: int = 10,
        path_length_m: float = 20.0,
        vehicle_speed_mps: float = 2.0,
        timestep_s: float = 0.02,
        base_random_seed: int = 42,
        max_steps_per_episode: int = 2000,
        _sensor_noise_std: float = 0.0,
        **_: Any,
    ) -> None:
        if mujoco is None:  # pragma: no cover
            raise ImportError("MuJoCo is not installed. Install with: pip install mujoco>=3.1")
        self.num_envs = int(max(1, num_envs))
        self.n_obstacles = int(max(1, n_obstacles))
        self.path_length_m = float(path_length_m)
        self.vehicle_speed_mps = float(vehicle_speed_mps)
        self.timestep_s = float(timestep_s)
        self.base_random_seed = int(base_random_seed)
        self.max_steps_per_episode = int(max_steps_per_episode)
        self._sn = float(_sensor_noise_std or 0.0)
        self._slots: list[tuple[SensorConfig | None, dict | None]] = [
            (None, None) for _ in range(self.num_envs)
        ]
        self._gen: int = 0
        self._crn_seeds: list[int] = _crn_seeds(self.base_random_seed, 0, N_CRN_SEEDS)
        xml = build_vehicle_mjcf(self.n_obstacles)
        self.model = mujoco.MjModel.from_xml_string(xml)
        self.data = mujoco.MjData(self.model)
        self._exp_cfg: dict = {}

    def set_experiment_config(self, cfg: dict) -> None:
        self._exp_cfg = dict(cfg or {})
        il = self._exp_cfg.get("inner_loop", {})
        mj: Dict[str, Any] = (il or {}).get("mujoco", {}) or {} if isinstance(il, dict) else {}
        if isinstance(mj, dict) and mj:
            self.n_obstacles = int(mj.get("n_obstacles", self.n_obstacles))
            self.path_length_m = float(mj.get("path_length_m", self.path_length_m))
            self.vehicle_speed_mps = float(mj.get("vehicle_speed_mps", self.vehicle_speed_mps))
            self.timestep_s = float(mj.get("timestep_s", self.timestep_s))
            if mj.get("base_random_seed") is not None:
                self.base_random_seed = int(mj.get("base_random_seed", 42))
        rt = self._exp_cfg.get("runtime", {})
        if not isinstance(rt, dict):
            rt = {}
        if rt.get("base_random_seed") is not None:
            self.base_random_seed = int(rt["base_random_seed"])
        if rt.get("n_obstacles") is not None:
            self.n_obstacles = int(rt["n_obstacles"])
        if rt.get("path_length_m") is not None:
            self.path_length_m = float(rt["path_length_m"])
        if rt.get("vehicle_speed_mps") is not None:
            self.vehicle_speed_mps = float(rt["vehicle_speed_mps"])
        if rt.get("timestep_s") is not None:
            self.timestep_s = float(rt["timestep_s"])

    def reconfigure_sensors(self, env_idx: int, config: SensorConfig, sensor_models: dict) -> None:
        if env_idx < 0 or env_idx >= self.num_envs:
            raise IndexError("env_idx out of range")
        self._slots[env_idx] = (config, dict(sensor_models))

    def run_rollouts(
        self,
        n_episodes: int,
        rng: np.random.Generator,
        sensor_noise_std: float = 0.0,
        generation: int = 0,
    ) -> list[EvalMetrics]:
        _ = (rng, sensor_noise_std)
        gen = int(generation)
        # Refresh CRN seed pool when generation advances.
        # All candidates in the same generation share this pool.
        if gen != self._gen:
            self._gen = gen
            self._crn_seeds = _crn_seeds(self.base_random_seed, gen, N_CRN_SEEDS)

        out: list[EvalMetrics] = []
        for i in range(self.num_envs):
            cfg, sm = self._slots[i]
            if cfg is None or sm is None:
                out.append(_zero_metrics(n_episodes, self.n_obstacles))
                continue
            if not cfg.active_sensors():
                out.append(_zero_metrics(n_episodes, self.n_obstacles))
                continue
            out.append(self._rollout_one(cfg, sm, n_episodes))
        return out

    def _rollout_one(self, config: SensorConfig, sensor_models: dict, n_episodes: int) -> EvalMetrics:
        scfg = _slot_to_config(config)
        sspec = get_sensor_specs(self._exp_cfg)
        n_ep = max(1, n_episodes)

        # CRN: cycle through this generation\'s seed pool.
        # Each episode gets a unique, generation-specific seed.
        ep_seeds = [self._crn_seeds[i % N_CRN_SEEDS] for i in range(n_ep)]

        sim_c = {
            "path_length_m": self.path_length_m,
            "vehicle_speed_mps": self.vehicle_speed_mps,
            "timestep_s": self.timestep_s,
            "n_obstacles": self.n_obstacles,
        }
        covs: list[float] = []
        ndets: list[float] = []
        mdd: list[float] = []
        ftm: list[float] = []
        pslots: list[dict] = []
        n_obs_l: list[int] = []
        t_ep: list[float] = []

        for ep, ep_seed in enumerate(ep_seeds):
            pos = generate_obstacles(
                self.n_obstacles,
                self.path_length_m,
                ep_seed,
            )
            m = run_episode(
                self.model,
                self.data,
                scfg,
                pos,
                sim_c,
                sspec,
                np.random.default_rng(int(ep_seed) + 17 * ep),
            )
            covs.append(float(m["coverage_fraction"]))
            ndets.append(float(m["n_detected"]))
            mdd.append(float(m["mean_detection_distance"]))
            ft = m["first_detection_times"]
            T = float(m["episode_duration_s"])
            t_ep.append(T)
            vals: list[float] = []
            for f in ft:
                if f is None:
                    vals.append(T)
                else:
                    vals.append(float(f))
            ftm.append(float(np.mean(vals)) if vals else T)
            pslots.append(dict(m.get("per_slot_first_hits", {})))
            n_obs_l.append(int(m.get("n_obstacles", self.n_obstacles)))

        pmean = {s: 0.0 for s in SLOT_NAMES}
        for p in pslots:
            for k, v in p.items():
                if k in pmean:
                    pmean[k] += float(v) / n_ep

        return EvalMetrics(
            collision_rate=0.0,
            blind_spot_fraction=0.0,
            mean_goal_success=float(np.mean(ndets) / max(float(self.n_obstacles), 1.0)),
            n_episodes=n_ep,
            t_det_s=float(np.mean(ftm)),
            t_det_s_p95=float(np.percentile(np.asarray(ftm, dtype=np.float64), 95.0)),
            episode_time_s=float(np.mean(t_ep)) if t_ep else 0.0,
            detection_miss_rate=1.0 - float(np.mean(ndets) / max(float(self.n_obstacles), 1.0)),
            coverage_fraction=float(np.mean(covs)),
            n_detected=float(np.mean(ndets)),
            n_obstacles=float(np.mean(n_obs_l)),
            mean_detection_distance_m=float(np.mean(mdd)) if mdd else 0.0,
            first_detection_time_mean=float(np.mean(ftm)),
            per_slot_first_hits={k: float(v) for k, v in pmean.items()},
        )


def _zero_metrics(n_episodes: int, n_obstacles: int) -> EvalMetrics:
    T = 1.0
    return EvalMetrics(
        collision_rate=0.0,
        blind_spot_fraction=1.0,
        mean_goal_success=0.0,
        n_episodes=max(0, n_episodes),
        episode_time_s=T,
        coverage_fraction=0.0,
        n_detected=0.0,
        n_obstacles=float(n_obstacles),
        mean_detection_distance_m=0.0,
        first_detection_time_mean=T,
        per_slot_first_hits={s: 0.0 for s in SLOT_NAMES},
    )
'''

path = Path("/content/sensor_placement_opt_MUJOCO_ver/sensor_opt/inner_loop/mujoco_env_manager.py")
path.write_text(NEW_SRC, encoding="utf-8")
print(f"✅ Written {path.stat().st_size} bytes")
print("\nCRN seed pool preview (gen=0, base=42):")
import numpy as np
rng = np.random.default_rng([42, 0])
print(" ", [int(s) for s in rng.integers(0, 2**31 - 1, size=10)])
print("\nCRN seed pool preview (gen=1, base=42):")
rng = np.random.default_rng([42, 1])
print(" ", [int(s) for s in rng.integers(0, 2**31 - 1, size=10)])
print("\n✅ Pools are different across generations — CRN is working.")
print("Now run Cell 9.")

✅ Written 9685 bytes

CRN seed pool preview (gen=0, base=42):
  [191664963, 1662057957, 1405681631, 942484272, 929893137, 1843824992, 184566854, 1497586438, 432652533, 202244314]

CRN seed pool preview (gen=1, base=42):
  [1370068125, 1704034044, 965090078, 528192194, 1975775578, 868498938, 1766788444, 1358831403, 1938359164, 1752763708]

✅ Pools are different across generations — CRN is working.
Now run Cell 9.


In [14]:
# ── Cell 9 ─ Dual optimisation runs + manifest ─────────────────────────────────
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import yaml

TARGET = Path("/content/sensor_placement_opt_MUJOCO_ver")
os.chdir(TARGET)
if str(TARGET) not in sys.path:
    sys.path.insert(0, str(TARGET))

from sensor_opt.config.specs import prepare_experiment_config

with open("configs/mujoco_vehicle.yaml", encoding="utf-8") as f:
    raw = yaml.safe_load(f)


def _snapshot_run_dirs():
    return {p.resolve() for p in Path("results").glob("mujoco_*") if p.is_dir()}


OPTIMIZATION_MODES_TO_RUN = ["safety_first", "efficiency"]
for _m in OPTIMIZATION_MODES_TO_RUN:
    assert _m in _LOSS_PRESETS, f"Unknown mode {_m}; add it to _LOSS_PRESETS in Cell 4"


def _popsize_for_mode(mode: str) -> int:
    by = globals().get("CMA_POPSIZE_BY_MODE")
    if isinstance(by, dict):
        return int(by.get(mode, CMA_POPSIZE))
    return int(CMA_POPSIZE)


dirs_before = _snapshot_run_dirs()
DUAL_RUN_MANIFEST = []

for mode in OPTIMIZATION_MODES_TO_RUN:
    loss_weights = _LOSS_PRESETS[mode]
    popsz = _popsize_for_mode(mode)

    overrides = {
        "runtime": {
            "trial_type": TRIAL_TYPE,
            "PATH_LENGTH_M": PATH_LENGTH_M,
            "VEHICLE_SPEED_MPS": VEHICLE_SPEED_MPS,
            "N_OBSTACLES": N_OBSTACLES,
            "BASE_RANDOM_SEED": BASE_RANDOM_SEED,
            "MAX_HARDWARE_BUDGET_USD": MAX_HARDWARE_BUDGET_USD,
            "SENSOR_COSTS_USD": SENSOR_COSTS_USD,
            "MAX_SENSOR_COUNT": MAX_SENSOR_COUNT,
            "MAX_SENSOR_COUNTS": MAX_SENSOR_COUNTS,
            "OPTIMIZATION_MODE": mode,
            "HARDWARE_PLATFORM": HARDWARE_PLATFORM,
            "max_generations": MAX_GENERATIONS,
            "n_episodes": N_EPISODES,
            "n_obstacles": N_OBSTACLES,
        },
        "cma": {
            "max_generations": MAX_GENERATIONS,
            "population_size": popsz,
            "use_recommended_popsize": False,
        },
        "inner_loop": {
            "n_episodes": N_EPISODES,
            "mujoco": {"n_obstacles": N_OBSTACLES},
        },
        "loss": {"trial_weight_overrides": loss_weights},
    }

    full = prepare_experiment_config(raw, overrides)
    out_cfg = Path(f"configs/_notebook_active_{mode}.yaml")
    out_cfg.write_text(
        yaml.dump(full, default_flow_style=False, sort_keys=False),
        encoding="utf-8",
    )

    print(f"\n{'=' * 60}\nMode: {mode}  |  popsize={popsz}  |  weights: {loss_weights}")
    print(f"Wrote {out_cfg}")
    print(f"Platform: {HARDWARE_PLATFORM['name']}  |  Bandwidth: {HARDWARE_PLATFORM['memory_bandwidth_gbps']} GB/s")

    cmd = [
        sys.executable,
        "-u",
        "-m",
        "sensor_opt.run_experiment",
        "--config",
        str(out_cfg),
        "--no-mlflow",
    ]
    env = os.environ.copy()
    env.update({"PYTHONUNBUFFERED": "1", "FORCE_COLOR": "0", "NO_COLOR": "1"})

    print(f"Launching: {' '.join(cmd)}\n{'-' * 60}")
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    elapsed = time.time() - t0

    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

    dirs_after = _snapshot_run_dirs()
    new_dirs = dirs_after - dirs_before
    if len(new_dirs) != 1:
        raise RuntimeError(
            f"Expected exactly one new results folder for mode={mode}, got {len(new_dirs)}: {new_dirs}"
        )
    run_dir = next(iter(new_dirs))
    dirs_before = dirs_after

    csv_path = run_dir / "generations.csv"
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing generations.csv under {run_dir}")

    DUAL_RUN_MANIFEST.append(
        {
            "mode": mode,
            "population_size": popsz,
            "run_dir": str(run_dir),
            "generations_csv": str(csv_path),
            "elapsed_s": elapsed,
        }
    )
    print(f"✅ [{mode}] Done in {elapsed / 60:.1f} min  →  {run_dir.name}")

manifest_path = Path("results/_notebook_dual_run_manifest.json")
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({"runs": DUAL_RUN_MANIFEST}, f, indent=2)
print(f"\n📎 Manifest saved: {manifest_path}")
print("Dual-run summary:")
for row in DUAL_RUN_MANIFEST:
    print(f"  • {row['mode']:14s}  pop={row['population_size']:<3d}  {Path(row['run_dir']).name}")


Wrote configs/_notebook_active.yaml
Mode: safety_first  |  Weights: {'w_acc': 0.6, 'w_lat': 0.35, 'w_cost': 0.05}
Platform: Apple M5 (24 GB)  |  Bandwidth: 153.6 GB/s
Launching: /usr/bin/python3 -u -m sensor_opt.run_experiment --config configs/_notebook_active.yaml --no-mlflow
[Experiment] name    : mujoco_multi_objective
[Experiment] mode    : mujoco
[Experiment] seed    : 42
[Experiment] config  : configs/_notebook_active.yaml
[Experiment] Using MuJoCo evaluator
[CMA-ES] Vector dimension: 5 (5 sensor slots × 1 params)
[CMA-ES] popsize=20  sigma0=0.2500  max_generations=100
[CMA-ES] Initial seed: n_active=5 | collision_rate=0.000 (~0/3 high-coll ep.) | blind=0.000
Gen    1 | gen_best=0.0637 | all_best=0.0637 | σ=0.1956 | active=5 | $24 | top_front_edge_l=camera($2); top_front_edge_r=camera($2); front_face_center=lidar($16); side_front_edge_l=camera($2); side_front_edge_r=camera($2)
[Diag G1] best-of-pop: n_coll~0 / 3 | blind=0.000 | n_active=5 | uncovered_slots~0 | oob=0 (decode bound

In [ ]:
# ── Cell 10 ─ Compare convergence: safety_first vs efficiency ───────────────────
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

TARGET = Path("/content/sensor_placement_opt_MUJOCO_ver")
os.chdir(TARGET)
if str(TARGET) not in sys.path:
    sys.path.insert(0, str(TARGET))


def _load_manifest():
    if "DUAL_RUN_MANIFEST" in globals() and len(globals()["DUAL_RUN_MANIFEST"]) >= 2:
        return globals()["DUAL_RUN_MANIFEST"]
    mp = Path("results/_notebook_dual_run_manifest.json")
    if mp.is_file():
        data = json.loads(mp.read_text(encoding="utf-8"))
        return data.get("runs", [])
    return []


def _pick_column(df: pd.DataFrame, candidates: list[str]):
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in df.columns:
            return name
        low = name.lower()
        if low in cols_lower:
            return cols_lower[low]
    return None


manifest = _load_manifest()
assert len(manifest) >= 2, (
    "Need two runs — execute Cell 9 first (or reload manifest). Got: "
    + str([m.get("mode") for m in manifest])
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for i, row in enumerate(manifest):
    csv_path = Path(row["generations_csv"])
    mode = row.get("mode", csv_path.parent.name)
    assert csv_path.is_file(), f"Missing CSV: {csv_path}"

    df = pd.read_csv(csv_path)
    gen_col = _pick_column(df, ["generation", "gen", "Gen", "Generation"])
    if gen_col is None:
        gen_col = df.columns[0]

    best_col = _pick_column(
        df,
        ["all_best", "best_so_far", "best_loss", "best", "fitness_best"],
    )
    gen_best_col = _pick_column(df, ["gen_best", "generation_best"])

    if best_col is None and gen_best_col is not None:
        best_col = gen_best_col
        gen_best_col = None

    if best_col is None:
        raise ValueError(
            f"Could not find objective column in {csv_path}; columns={list(df.columns)}"
        )

    g = pd.to_numeric(df[gen_col], errors="coerce")
    y = pd.to_numeric(df[best_col], errors="coerce")
    lbl = "best-so-far" if best_col.lower().startswith("all") or "best_so" in best_col.lower() else "objective"
    ax.plot(g, y, label=f"{mode} ({lbl})", color=colors[i % len(colors)], linewidth=2)

    if gen_best_col is not None:
        y2 = pd.to_numeric(df[gen_best_col], errors="coerce")
        ax.plot(
            g,
            y2,
            linestyle="--",
            alpha=0.65,
            color=colors[i % len(colors)],
            label=f"{mode} (gen best)",
        )

ax.set_xlabel("Generation")
ax.set_ylabel("Objective value")
ax.set_title("CMA-ES convergence: dual optimization modes")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Sources:")
for row in manifest:
    print(f"  • {row['mode']}: {row['generations_csv']}")

In [ ]:
import os, json
from pathlib import Path

# Load the best config from the most recent results folder (after Cell 9 dual run,
# this is typically the second mode: efficiency — see results/_notebook_dual_run_manifest.json).
run_dir = sorted(Path("results").glob("mujoco_*"), key=lambda p: p.name, reverse=True)[0]
config = json.loads((run_dir / "config.json").read_text())
print(f"Visualising: {run_dir.name}")

# Check what's available for rendering
import mujoco
print(f"MuJoCo version: {mujoco.__version__}")
print(f"Render backends: {mujoco.mjtRndFlag.__members__.keys()}")

In [ ]:
# ── (a) Headless GL setup — MUST run before any mujoco import ─────────────────
import subprocess, os

# Install EGL (or fall back to osmesa for pure software rendering)
subprocess.run(
    ["apt-get", "install", "-y", "-q", "libegl1-mesa", "libegl1-mesa-dev"],
    check=False,   # non-fatal if already installed
)
# Override whatever a previous cell may have set
os.environ["MUJOCO_GL"] = "egl"

# If EGL still fails, swap to osmesa (slower but always works on Colab CPU):
# subprocess.run(["apt-get", "install", "-y", "-q", "libosmesa6-dev"], check=False)
# os.environ["MUJOCO_GL"] = "osmesa"

In [ ]:
# ── Cell 12 ─ MuJoCo simulation video (v2) ───────────────────────────────────
# Renders one episode with the best sensor config and saves / displays an MP4.
# Uses the newest results folder — after a dual Cell 9 run that is usually `efficiency`;
# pick another folder manually if you want `safety_first`.
#
# Run AFTER Cell 11.  Requires patch cells through env-manager rewrite still applied
# (i.e. the patched sensor_opt source in /content/sensor_placement_opt_MUJOCO_ver).
# ─────────────────────────────────────────────────────────────────────────────

# ── (a) Deps ──────────────────────────────────────────────────────────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import imageio
except ImportError:
    _pip("imageio[ffmpeg]", "imageio-ffmpeg")
    import imageio

# ── (b) Force EGL before any mujoco import (headless Colab) ──────────────────
import os
os.environ.setdefault("MUJOCO_GL", "egl")   # switch to "osmesa" if egl errors

# ── (c) Stdlib / third-party imports ─────────────────────────────────────────
import json, inspect, time
from pathlib import Path

import numpy as np
import mujoco

# ── (d) Repo path ─────────────────────────────────────────────────────────────
TARGET = Path("/content/sensor_placement_opt_MUJOCO_ver")
os.chdir(TARGET)
if str(TARGET) not in sys.path:
    sys.path.insert(0, str(TARGET))

# ── (e) sensor_opt imports ────────────────────────────────────────────────────
from sensor_opt.config.specs      import prepare_experiment_config
from sensor_opt.simulation.mjcf   import build_vehicle_mjcf, SLOT_NAMES   # ← correct name
from sensor_opt.simulation.obstacles import generate_obstacles

# ── (f) Locate most-recent run ────────────────────────────────────────────────
results_dirs = sorted(
    Path("results").glob("mujoco_*"),
    key=lambda p: p.name,
    reverse=True,
)
assert results_dirs, "No results directory found — run Cell 9 first."
run_dir = results_dirs[0]
print(f"Run : {run_dir.name}")

# ── (g) Load experiment config ────────────────────────────────────────────────
with open(run_dir / "config.json", encoding="utf-8") as f:
    full_cfg = prepare_experiment_config(json.load(f))

rt           = full_cfg.get("runtime", {})
n_obstacles  = int(rt.get("n_obstacles",      N_OBSTACLES))
path_len     = float(rt.get("PATH_LENGTH_M",  PATH_LENGTH_M))
speed_mps    = float(rt.get("VEHICLE_SPEED_MPS", VEHICLE_SPEED_MPS))
timestep_s   = float(full_cfg.get("inner_loop", {}).get("mujoco", {}).get("timestep_s", 0.02))

# ── (h) Recover best sensor layout ───────────────────────────────────────────
checkpoints = sorted(
    (run_dir / "checkpoints").glob("checkpoint_gen*.json"),
    key=lambda p: p.name,
)
if checkpoints:
    with open(checkpoints[-1], encoding="utf-8") as f:
        best_cfg = json.load(f).get("best_config")
else:
    with open(run_dir / "evaluated_pool.json", encoding="utf-8") as f:
        pool = json.load(f)
    objs    = [sum(r.get("objectives", {}).values()) for r in pool]
    best_cfg = pool[int(np.argmin(objs))].get("config") if objs else None

assert best_cfg, "Could not recover best_config — check results directory."

# Build the slot→type dict the same way mujoco_env_manager does
scfg: dict[str, str] = {s: "disabled" for s in SLOT_NAMES}
for s in best_cfg.get("sensors", []):
    slot = s.get("slot")
    stype = s.get("type", "disabled")
    if slot in scfg:
        scfg[slot] = stype

print("Sensor config:", scfg)

# ── (i) Build MuJoCo model ────────────────────────────────────────────────────
# build_vehicle_mjcf only needs the max number of obstacles (pool size in XML)
xml    = build_vehicle_mjcf(n_obstacles)
model  = mujoco.MjModel.from_xml_string(xml)
data   = mujoco.MjData(model)

print(f"Model: {model.nbody} bodies | {model.nmocap} mocap | {model.njnt} joints | dt={model.opt.timestep:.4f}s")

# ── (j) Introspect model structure ───────────────────────────────────────────
# Print body/joint/mocap names so we know the exact identifiers at runtime.
body_names  = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY,  i) or f"body_{i}"
               for i in range(model.nbody)]
joint_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i) or f"joint_{i}"
               for i in range(model.njnt)]

print("Bodies :", body_names)
print("Joints :", joint_names)

# ── Locate vehicle body ───────────────────────────────────────────────────────
# Try common names; fall back to the first non-world body.
_VEHICLE_CANDIDATES = ["vehicle", "car", "robot", "agent", "ego"]
vehicle_body_id = None
for name in _VEHICLE_CANDIDATES:
    bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
    if bid >= 0:
        vehicle_body_id = bid
        vehicle_name    = name
        break
if vehicle_body_id is None:
    # Fall back: first body that isn't "world"
    vehicle_body_id = 1
    vehicle_name    = body_names[1] if len(body_names) > 1 else "body_1"

print(f"Vehicle body: '{vehicle_name}' (id={vehicle_body_id})")

# ── Locate the vehicle's free/slide joint for position control ────────────────
# A freejoint stores [x, y, z, qw, qx, qy, qz] in qpos.
# A slide joint along X stores just [x].
vehicle_jnt_id = -1
vehicle_jnt_type = None
for i, jname in enumerate(joint_names):
    jtype = model.jnt_type[i]          # 0=free, 1=ball, 2=slide, 3=hinge
    if body_names[model.jnt_bodyid[i]] == vehicle_name:
        vehicle_jnt_id   = i
        vehicle_jnt_type = jtype
        break

print(f"Vehicle joint: id={vehicle_jnt_id} type={vehicle_jnt_type} "
      f"({'free' if vehicle_jnt_type==0 else 'slide/other'})")

# ── Locate obstacle mocap bodies ──────────────────────────────────────────────
# Mocap bodies are written to data.mocap_pos[mocap_idx].
# We need the mapping: mocap_body_name → mocap_idx
mocap_ids = []   # ordered list of mocap body ids we will use for obstacles
for bid in range(model.nbody):
    if model.body_mocapid[bid] >= 0:      # -1 means not a mocap body
        bname = body_names[bid]
        mocap_ids.append((bid, model.body_mocapid[bid], bname))

print(f"Mocap bodies ({len(mocap_ids)}): {[(n, mid) for _, mid, n in mocap_ids]}")

# ── (k) Generate obstacle layout ─────────────────────────────────────────────
obs_positions = generate_obstacles(n_obstacles, path_len, rng_seed=42_000)
print(f"Obstacles: {len(obs_positions)} at seed 42_000")

# ── (l) Helper: set scene state at time t ─────────────────────────────────────
def _set_scene(t: float):
    """Move vehicle along +X and place obstacles; update kinematics."""
    x_vehicle = speed_mps * t

    # -- Move vehicle --
    if vehicle_jnt_id >= 0:
        qadr = model.jnt_qposadr[vehicle_jnt_id]
        if vehicle_jnt_type == 0:              # freejoint: x y z qw qx qy qz
            data.qpos[qadr]     = x_vehicle   # x
            data.qpos[qadr + 1] = 0.0         # y
            data.qpos[qadr + 2] = 0.0         # z (ground level — adjust if needed)
            data.qpos[qadr + 3] = 1.0         # qw  (identity quaternion)
            data.qpos[qadr + 4] = 0.0
            data.qpos[qadr + 5] = 0.0
            data.qpos[qadr + 6] = 0.0
        elif vehicle_jnt_type == 2:            # slide along X
            data.qpos[qadr] = x_vehicle
    else:
        # No joint found: try xpos directly (only works if body has no parent joint)
        data.xpos[vehicle_body_id, 0] = x_vehicle

    # -- Place obstacles via mocap --
    for i, (bid, mid, bname) in enumerate(mocap_ids):
        if i < len(obs_positions):
            x, y, z = obs_positions[i]
            data.mocap_pos[mid] = [x, y, z]
        else:
            # Park unused mocap slots far away
            data.mocap_pos[mid] = [999.0, 999.0, -1.0]

    mujoco.mj_kinematics(model, data)    # update site/body xpos without physics

# ── (m) Video parameters ──────────────────────────────────────────────────────
VIDEO_FPS     = 30
SIM_DURATION  = path_len / speed_mps          # seconds
FRAME_SKIP    = max(1, int(round(1.0 / (VIDEO_FPS * timestep_s))))
WIDTH, HEIGHT = 640, 480
N_STEPS       = int(SIM_DURATION / timestep_s)

print(f"\nSimulating {SIM_DURATION:.1f}s | {N_STEPS} steps | "
      f"capturing every {FRAME_SKIP} steps → ~{N_STEPS//FRAME_SKIP} frames")

# ── (n) Render loop ────────────────────────────────────────────────────────────
renderer = mujoco.Renderer(model, height=HEIGHT, width=WIDTH)

# Camera: birds-eye angled view looking along the path
camera           = mujoco.MjvCamera()
camera.type      = mujoco.mjtCamera.mjCAMERA_FREE
camera.lookat[:] = [path_len / 2, 0.0, 0.3]
camera.distance  = path_len * 0.7
camera.elevation = -28.0
camera.azimuth   = 90.0

opt = mujoco.MjvOption()
opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True

mujoco.mj_resetData(model, data)
_set_scene(0.0)   # initialise positions before first render

frames = []
t0 = time.time()

for step in range(N_STEPS):
    t_sim = step * timestep_s
    _set_scene(t_sim)

    if step % FRAME_SKIP == 0:
        renderer.update_scene(data, camera=camera, scene_option=opt)
        frame = renderer.render()
        frames.append(frame.copy())

renderer.close()
print(f"Captured {len(frames)} frames in {time.time()-t0:.1f}s")

# ── (o) Encode MP4 ────────────────────────────────────────────────────────────
video_path = run_dir / "simulation_best_config.mp4"
with imageio.get_writer(str(video_path), fps=VIDEO_FPS, codec="libx264", quality=8) as w:
    for frame in frames:
        w.append_data(frame)

print(f"✅ Saved → {video_path}  ({video_path.stat().st_size/1e6:.1f} MB)")

# ── (p) Display inline ────────────────────────────────────────────────────────
try:
    from IPython.display import Video, display
    display(Video(str(video_path), embed=True, width=900))
except Exception:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    idxs   = [0, len(frames)//2, len(frames)-1]
    labels = ["Start", "Mid", "End"]
    for ax, idx, lbl in zip(axes, idxs, labels):
        ax.imshow(frames[idx])
        ax.set_title(f"{lbl} — frame {idx}")
        ax.axis("off")
    fig.suptitle(f"Best config preview — {run_dir.name}", fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f"(Full video saved to: {video_path})")